Finding a function $u=\sin(\pi x)\sin(\pi y) + \sin(\frac{\pi}{\omega}x)\sin(\frac{\pi}{\omega}y)$ that looks the way I want in the unit square.

In [45]:
# Import any packages needed
from ngsolve import *
from ngsolve.webgui import Draw
import matplotlib.pyplot as plt
import numpy as np

In [46]:
# Domain is unit square, create initial mesh
N = 16 
mesh = Mesh(unit_square.GenerateMesh(maxh=1/N))
mesh.nv, mesh.ne
Draw(mesh)

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

BaseWebGuiScene

In [ ]:
# mesh.Refine()
# fine_mesh = Mesh(mesh.ngmesh.Copy())
# Draw(mesh)
# Draw(fine_mesh)

In [51]:
fes = H1(mesh, order=1, dirichlet="left|right")
fes.ndof


339

In [ ]:
# do a smart copy to save this fes as our coarse one
cfes = fes
cfes.ndof


339

In [54]:
fes.mesh.Refine()
ffes.ndof
# # mesh, make fes, save it as coarse, refine, make prolong, save new refined as fine

5025

In [55]:
cfes.ndof

1289

In [ ]:
help(fes)

In [ ]:
cfes = H1(coarse_mesh, order=1, dirichlet="left|right")
cfes.ndof
cfes.mesh.levels

In [ ]:
# Working with H1-conforming piecewise linear finite elements
# boundary conditions are Dirichlet on the left and right and default=Neumann on the other 2 sides
fes = H1(mesh, order=1, dirichlet="left|right")
fes.ndof

# Set test and trial spaces
u = fes.TrialFunction()
phi = fes.TestFunction()

# Solution will be stored as a grid function
gfu = GridFunction(fes)
gfu.Set(1, BND) # this sets both Dirichlet boundary components to 1, might want to modify later
Draw(gfu)

In [ ]:
# Coefficient function for K(x,y)
Kxy = CoefficientFunction(1+x*x+y*y)
# Kxy = CoefficientFunction(3)
sigma = 0.5

# Bilinear form
a = BilinearForm(fes)
a += Kxy*grad(u)*grad(phi)*dx+sigma*sigma*u*phi*dx
a.Assemble()

# Right hand side
f = LinearForm(fes)
f.Assemble()

print(a.mat)
# print(f.vec)

In [ ]:
# To see the K(x,y) function, we plot it on our mesh
# Draw(Kxy, mesh)
# Draw the initial state of the grid function 
# Draw(gfu)

# Solve the PDE
c = Preconditioner(a,"local")
c.Update()
solvers.BVP(bf=a, lf=f, gf=gfu, pre=c)
Draw(gfu)

In [ ]:
# WEEK 1 Task
# Test a nice u and find f then work back to it
# what about u(x,y)=1-3x^2+2x^3 ?
plant = CoefficientFunction(1-3*x*x+2*x*x*x)
Draw(plant, mesh)

# Code the right hand side given the u selected
# f = partial w.r.t. x (k(x,y))
# f = - div (k grad u) + sigma^2 u 
# = - d/dx(k grad u) - d/dy(k grad u) + sigma^2 u
fakef = 25/4 -12*x + 12*x*y-12*x*y**2 + 69/4 *x**2 -12*x**2*y - 47/2 *x**3 + 6*y**2
Draw(fakef,mesh)
newf = LinearForm(fes)
newf += fakef*phi*dx
newf.Assemble()

# Solve it
gfu2 = GridFunction(fes)
gfu2.Set(1, BND)
c = Preconditioner(a,"local")
c.Update()
solvers.BVP(bf=a, lf=newf, gf=gfu2, pre=c)
Draw(gfu2)